In [ ]:
# Enable autoreload - automatically reimport modules when they change
# This is perfect for development with editable installs (pip install -e .)
%load_ext autoreload
%autoreload 2

print("✅ Autoreload enabled - changes to src/arcflow/ will be picked up automatically!")
print("💡 Make sure you're using venv-dev (editable install) for live code updates")

✅ Autoreload enabled - changes to src/arcflow/ will be picked up automatically!
💡 Make sure you're using venv-dev (editable install) for live code updates


In [5]:
spark.stop()

In [1]:
import os
from pyspark.sql import SparkSession

# Stop existing session if any
try:
    spark.stop()  # noqa: F821
    print("🛑 Stopped existing Spark session")
except:
    pass

# Configure Delta Lake with builder pattern
# The key is to use configure_spark_with_delta_pip() from delta.pip_utils
from delta import configure_spark_with_delta_pip
warehouse_dir = os.path.abspath(r'C:\Users\milescole\source\ArcFlow\dev\storage\lakehouse')
builder = (SparkSession.builder
    .appName("ArcFlow Example")
    .config("spark.driver.memory", "4g")
    .master("local[*]")
    .config("spark.ui.enabled", "false")
    .config("spark.sql.warehouse.dir", warehouse_dir)
    .config(
        "spark.hadoop.javax.jdo.option.ConnectionURL",
        "jdbc:derby:C:/Users/milescole/source/ArcFlow/dev/storage/lakehouse/metastore_db;create=true"
    )
    .config("spark.driver.host", "localhost")
    .config("spark.driver.bindAddress", "localhost")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0")
    .config("spark.sql.catalogImplementation", "hive")
    # for windows
    .config("spark.hadoop.io.native.lib.available", "false")
    .config("spark.hadoop.fs.file.impl.disable.cache", "true")
)

# This automatically configures all Delta settings and downloads JARs
spark = configure_spark_with_delta_pip(builder, extra_packages=["org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0", "com.microsoft.azure:azure-eventhubs-spark_2.12:2.3.22"]).getOrCreate()

print(f"✅ Spark session created with Delta Lake support")
print(f"   Spark version: {spark.version}")
print(f"   App name: {spark.sparkContext.appName}")
print(f"   Master: {spark.sparkContext.master}")

✅ Spark session created with Delta Lake support
   Spark version: 3.5.7
   App name: ArcFlow Example
   Master: local[*]


In [8]:
# reload pipeline_config to pick up changes to custom transformers
import importlib
import pipeline_config
importlib.reload(pipeline_config)
from pipeline_config import *

from arcflow import Controller

# Configure pipeline
config = {
    'streaming_enabled': True,
    'checkpoint_uri': r'C:\Users\milescole\source\ArcFlow\dev\storage\checkpoints',
    'landing_uri': r"C:\Users\milescole\source\LakeGen\dev\output_test\mcmillian",
    'trigger_interval': '1 seconds' # default if not set at table level
}

# Initialize controller
controller = Controller(
    spark=spark,
    config=config,
    table_registry=tables
)


Overwriting existing transformer: explode_message_payload
Overwriting existing transformer: explode_data
Overwriting existing transformer: silver_item
Overwriting existing transformer: silver_shipment
Overwriting existing transformer: silver_shipment_scan_event
Overwriting existing transformer: silver_order_transformer
Overwriting existing transformer: silver_customer_transformer
Overwriting existing transformer: cast_generated_at


In [12]:
from arcflow.pipelines.zone_pipeline import ZonePipeline

pipeline = ZonePipeline(
    spark=spark,
    zone="bronze",
    config={'streaming_enabled': True}
)

# Test the output - returns a batch or in-memory DataFrame
df_preview = pipeline.test_input(tables["shipment"], limit=10)

In [13]:
df_preview.show()

+----+--------------------+--------------------+---------+------+--------------------+-------------+
| key|               value|               topic|partition|offset|           timestamp|timestampType|
+----+--------------------+--------------------+---------+------+--------------------+-------------+
|NULL|{{2026-02-19T21:1...|es_a61f574d-aa40-...|        3|    65|2026-02-19 14:16:...|            1|
|NULL|{{2026-02-19T21:1...|es_a61f574d-aa40-...|        3|    66|2026-02-19 14:16:...|            1|
|NULL|{{2026-02-19T21:1...|es_a61f574d-aa40-...|        0|    66|2026-02-19 14:16:...|            1|
|NULL|{{2026-02-19T21:2...|es_a61f574d-aa40-...|        0|    67|2026-02-19 14:22:...|            1|
|NULL|{{2026-02-19T21:2...|es_a61f574d-aa40-...|        0|    68|2026-02-19 14:22:...|            1|
|NULL|{{2026-02-19T21:2...|es_a61f574d-aa40-...|        2|    73|2026-02-19 14:22:...|            1|
|NULL|{{2026-02-19T21:2...|es_a61f574d-aa40-...|        2|    74|2026-02-19 14:22:...|     

In [10]:
df_preview.select("data.*").show()

+-----------------------+-------------------+--------------------+--------------------+-----------------+--------------------+----------------+-------------------+-----------------------+--------------------+---------------------+-----------------+--------------------+------------------+--------------------+------+----------+------------+-----------------------------+------+--------------------+---------------+---------------+--------------------+-------------+--------------+------------------+---------------+----------------+------------+---------------+----------------------+-------------+--------------------+--------------------+--------------------+--------------------+------+------------------+-----+
|committed_delivery_date|current_facility_id|         customer_id|       customer_name|   declared_value| destination_address|destination_city|destination_country|destination_facility_id|destination_latitude|destination_longitude|destination_state|destination_zip_code|          distan

In [13]:
df = explode_message_payload2(df_preview)
df.show()

NameError: name 'explode_message_payload2' is not defined

In [70]:
df_preview.select("body.*").show()

+--------------------+--------------------+
|               _meta|                data|
+--------------------+--------------------+
|{2026-02-19T21:16...|[{2026-02-24T14:1...|
|{2026-02-19T21:22...|[{2026-02-23T14:2...|
|{2026-02-19T21:22...|[{2026-02-24T14:2...|
|{2026-02-19T21:22...|[{2026-02-21T14:2...|
|{2026-02-19T21:22...|[{2026-02-23T14:2...|
|{2026-02-19T21:22...|[{2026-02-23T14:2...|
|{2026-02-19T21:22...|[{2026-02-24T14:2...|
|{2026-02-19T21:22...|[{2026-02-24T14:2...|
|{2026-02-19T21:22...|[{2026-02-24T14:2...|
|{2026-02-19T21:22...|[{2026-02-22T14:2...|
|{2026-02-19T21:22...|[{2026-02-23T14:2...|
|{2026-02-19T21:23...|[{2026-02-21T14:2...|
|{2026-02-19T21:23...|[{2026-02-21T14:2...|
|{2026-02-19T21:23...|[{2026-02-23T14:2...|
|{2026-02-19T21:23...|[{2026-02-21T14:2...|
|{2026-02-19T21:23...|[{2026-02-24T14:2...|
|{2026-02-19T21:23...|[{2026-02-23T14:2...|
|{2026-02-19T21:23...|[{2026-02-20T14:2...|
|{2026-02-19T21:23...|[{2026-02-21T14:2...|
|{2026-02-19T21:23...|[{2026-02-

In [59]:
df_preview = pipeline.test_input(tables["shipment"])

In [68]:

bronze_input = df_preview.select("body.*").selectExpr("_meta", "explode(data) as data").show()
#bronze_output = bronze_input.select("data.*").show()

+--------------------+--------------------+
|               _meta|                data|
+--------------------+--------------------+
|{2026-02-19T21:16...|{2026-02-24T14:16...|
|{2026-02-19T21:16...|{2026-02-22T14:16...|
|{2026-02-19T21:16...|{2026-02-24T14:16...|
|{2026-02-19T21:16...|{2026-02-22T14:16...|
|{2026-02-19T21:16...|{2026-02-24T14:16...|
|{2026-02-19T21:16...|{2026-02-20T14:16...|
|{2026-02-19T21:16...|{2026-02-22T14:16...|
|{2026-02-19T21:16...|{2026-02-23T14:16...|
|{2026-02-19T21:16...|{2026-02-23T14:16...|
|{2026-02-19T21:16...|{2026-02-21T14:16...|
|{2026-02-19T21:16...|{2026-02-24T14:16...|
|{2026-02-19T21:16...|{2026-02-24T14:16...|
|{2026-02-19T21:16...|{2026-02-23T02:16...|
|{2026-02-19T21:16...|{2026-02-22T14:16...|
|{2026-02-19T21:16...|{2026-02-23T02:16...|
|{2026-02-19T21:16...|{2026-02-23T14:16...|
|{2026-02-19T21:16...|{2026-02-25T02:16...|
|{2026-02-19T21:16...|{2026-02-23T14:16...|
|{2026-02-19T21:16...|{2026-02-21T14:16...|
|{2026-02-19T21:16...|{2026-02-2

In [38]:
df = silver_shipment(df_preview)

AnalysisException: Can only star expand struct data types. Attribute: `ArrayBuffer(data)`.; line 1 pos 0

In [8]:
df = (
        spark.readStream.format('kafka')
        .option('kafka.bootstrap.servers', 'esehmtcykdoc38uf8c3x3s2s.servicebus.windows.net:9093')
        .option('subscribe', 'es_c5d2895b-e48f-41b9-9842-c7bdfde79484')
        .option('kafka.security.protocol', 'SASL_SSL')
        .option('kafka.sasl.mechanism', 'PLAIN')
        .option('kafka.sasl.jaas.config', 'org.apache.kafka.common.security.plain.PlainLoginModule required username="$ConnectionString" password="Endpoint=sb://esehmtcykdoc38uf8c3x3s2s.servicebus.windows.net/;SharedAccessKeyName=key_7455191e-0fa0-4a20-974f-a8ce093cb144;SharedAccessKey=LB6vzhSf203b+/VVq8vd1Oe+zwIcSZ5zq+AEhJmXD5Y=')
        .option('startingOffsets', 'earliest')
        .option('failOnDataLoss', 'false')
    )
df.load().printSchema()

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql.types import StringType
from pyspark.sql.dataframe import DataFrame
checkpoint = r"C:\Users\milescole\source\ArcFlow\dev\storage\checkpoints\kafka_test"
df = (
        spark.readStream.format('kafka')
        .option('kafka.bootstrap.servers', 'esehmtcypknmdp3i6ao83erb.servicebus.windows.net:9093')
        .option('subscribe', 'es_a61f574d-aa40-4d49-9794-466d9dde69db')
        .option('kafka.security.protocol', 'SASL_SSL')
        .option('kafka.sasl.mechanism', 'PLAIN')
        .option('kafka.sasl.jaas.config', 'org.apache.kafka.common.security.plain.PlainLoginModule required username="$ConnectionString" password="Endpoint=sb://esehmtcypknmdp3i6ao83erb.servicebus.windows.net/;SharedAccessKeyName=key_e447b74c-5906-473e-95b3-5456d3a7b3a2;SharedAccessKey=7QxQv+stMgS01UeDadEm6DeePEclVbRsy+AEhDJaC1Q=";')
        .option('startingOffsets', 'earliest')
        .option('failOnDataLoss', 'false')
        .load()
    )
decoded_df = df
query= (decoded_df.writeStream
      .format("memory")
      .queryName("kafka_debug2")
      .outputMode("append")
      .start())

In [41]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql.types import StringType
from pyspark.sql.dataframe import DataFrame
checkpoint = r"C:\Users\milescole\source\ArcFlow\dev\storage\checkpoints\kafka_test"
df = (
        spark.readStream.format('eventhubs')
        .option('eventhubs.connectionString', 'Endpoint=sb://esehmtcypknmdp3i6ao83erb.servicebus.windows.net/;SharedAccessKeyName=key_e447b74c-5906-473e-95b3-5456d3a7b3a2;SharedAccessKey=pw6+QKKi2bQVt2A870eNZNskYFZMhqIXJ+AEhE86eoI=;EntityPath=es_a61f574d-aa40-4d49-9794-466d9dde69db')
        .option('eventhubs.startingPosition', {
        "offset": "-1",
        "seqNo": -1,
        "enqueuedTime": None,
        "isInclusive": True,
    })
        .load()
    )
decoded_df = df
query= (decoded_df.writeStream
      .format("memory")
      .queryName("eh_debug")
      .outputMode("append")
      .start())

In [44]:
query.stop()

In [43]:
spark.sql("SELECT * FROM eh_debug").show(truncate=True)

+----+---------+------+--------------+------------+---------+------------+----------+----------------+
|body|partition|offset|sequenceNumber|enqueuedTime|publisher|partitionKey|properties|systemProperties|
+----+---------+------+--------------+------------+---------+------------+----------+----------------+
+----+---------+------+--------------+------------+---------+------------+----------+----------------+



In [12]:
from arcflow import Controller
#from arcflow.core.spark_session import create_spark_session

# Create Spark session
#spark = create_spark_session(app_name="MyELT")

# Configure pipeline
config = {
    'streaming_enabled': True,
    'checkpoint_uri': r'C:\Users\milescole\source\ArcFlow\dev\storage\checkpoints',
    'landing_uri': r"C:\Users\milescole\source\LakeGen\dev\output_test\mcmillian",
    'trigger_interval': '1 seconds' # default if not set at table level
}

# Initialize controller
controller = Controller(
    spark=spark,
    config=config,
    table_registry=tables
)


In [4]:
# Run full pipeline
controller.run_full_pipeline(zones=['bronze', 'silver'])

In [7]:
controller.stop_all()

In [8]:
spark.sql("select count(1) from bronze.shipment").limit(5).show()

+--------+
|count(1)|
+--------+
|   18395|
+--------+



## Check Stream Status

After running the pipeline, you can check the status of all active streams:

In [6]:
controller.get_status()

{'bronze_item_stream': {'active': True,
  'id': 'f758086b-45f4-4b36-b425-eba81c2be3a8',
  'runId': '9a0efc51-985b-4cb1-8662-f874997d2eec',
  'recent_progress': {'id': 'f758086b-45f4-4b36-b425-eba81c2be3a8',
   'runId': '9a0efc51-985b-4cb1-8662-f874997d2eec',
   'name': 'bronze_item_stream',
   'timestamp': '2026-02-18T21:44:30.001Z',
   'batchId': 4,
   'numInputRows': 0,
   'inputRowsPerSecond': 0.0,
   'processedRowsPerSecond': 0.0,
   'durationMs': {'latestOffset': 3, 'triggerExecution': 3},
   'stateOperators': [],
   'sources': [{'description': 'FileStreamSource[file:/C:/Users/milescole/source/LakeGen/dev/output_test/mcmillian/item]',
     'startOffset': {'logOffset': 3},
     'endOffset': {'logOffset': 3},
     'latestOffset': None,
     'numInputRows': 0,
     'inputRowsPerSecond': 0.0,
     'processedRowsPerSecond': 0.0}],
   'sink': {'description': 'DeltaSink[file:/C:/Users/milescole/source/ArcFlow/dev/storage/lakehouse/bronze.db/item]',
    'numOutputRows': -1}}},
 'bronze_sh

In [3]:
# Alternative: Check individual query status directly
# (useful if you only ran a single table)

from arcflow.pipelines.zone_pipeline import ZonePipeline

pipeline = ZonePipeline(spark, zone="bronze", config=config)
query = pipeline.process_table(tables["shipment"])

if query:
    print(f"Query Name: {query.name}")
    print(f"Query ID: {query.id}")
    print(f"Is Active: {query.isActive}")
    print(f"Status: {query.status}")
    
    # Get recent progress
    progress = query.recentProgress
    if progress:
        latest = progress[-1]
        print(f"\nLatest Batch:")
        print(f"  Batch ID: {latest.get('batchId')}")
        print(f"  Input Rows: {latest.get('numInputRows')}")
        print(f"  Processing Time: {latest.get('durationMs', {}).get('triggerExecution')} ms")

NameError: name 'config' is not defined

In [ ]:
# Wait for all availableNow triggers to complete
# (This is safe for notebooks - only blocks until availableNow streams finish)
controller.await_completion()

print("✅ All streams completed!")

## Quick Preview: Pre-Write Output

Use `ZonePipeline._apply_transformations()` to see exactly what will be written

In [9]:
from arcflow.pipelines.zone_pipeline import ZonePipeline

pipeline = ZonePipeline(
    spark=spark,
    zone="silver",
    config={'streaming_enabled': True}
)

# Test the output - returns a batch DataFrame
df_preview = pipeline.test_output(tables["shipment"])

df_preview.show()

+--------------------+----------------+-----------+--------------+-----------------------+-------------------+--------------------+--------------------+------------------+--------------------+----------------+-------------------+-----------------------+--------------------+---------------------+-----------------+--------------------+------------------+--------------------+------+----------+------------+-----------------------------+------+--------------------+---------------+---------------+-------------------+-------------+--------------+------------------+---------------+----------------+------------+---------------+----------------------+-------------+--------------------+--------------------+--------------------+--------------------+------+------------------+-----+---------------------+
|       enqueued_time|        producer|record_type|schema_version|committed_delivery_date|current_facility_id|         customer_id|       customer_name|    declared_value| destination_address|destin

## Process Single Table with availableNow Trigger

The `availableNow` trigger processes all currently available data and then stops - perfect for testing streaming pipelines!

In [ ]:
from arcflow.pipelines.zone_pipeline import ZonePipeline

# Create a streaming pipeline for bronze zone
pipeline = ZonePipeline(
    spark=spark,
    zone="bronze",
    config={
        'streaming_enabled': True,  # Enable streaming mode
        'checkpoint_uri': r'C:\Users\milescole\source\ArcFlow\dev\storage\checkpoints'
    }
)

query = pipeline.process_table(tables["shipment"])


if query:
    print(f"✅ Started streaming query: {query.name}")
    print(f"   Status: {query.status}")
    
    # Wait for the query to finish (availableNow will complete automatically)
    query.awaitTermination()
    
    print(f"\n✅ Query completed successfully!")
    print(f"   Final status: {query.status}")
else:
    print("❌ No query was started - check if shipment bronze zone is enabled")

✅ Started streaming query: bronze_shipment_stream
   Status: {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}

✅ Query completed successfully!
   Final status: {'message': 'Stopped', 'isDataAvailable': False, 'isTriggerActive': False}


# Process Silver Table

In [ ]:
from arcflow.pipelines.zone_pipeline import ZonePipeline

# Create a streaming pipeline for bronze zone
pipeline = ZonePipeline(
    spark=spark,
    zone="silver",
    config={
        'streaming_enabled': True,  # Enable streaming mode
        'checkpoint_uri': r'C:\Users\milescole\source\ArcFlow\dev\storage\checkpoints'
    }
)

query = pipeline.process_table(tables["shipment"])


if query:
    print(f"✅ Started streaming query: {query.name}")
    print(f"   Status: {query.status}")
    
    # Wait for the query to finish (availableNow will complete automatically)
    query.awaitTermination()
    
    print(f"\n✅ Query completed successfully!")
    print(f"   Final status: {query.status}")
else:
    print("❌ No query was started - check if shipment silver zone is enabled")

✅ Started streaming query: silver_shipment_stream
   Query Name: silver_shipment_stream
   Status: {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}

✅ Query completed successfully!
   Final status: {'message': 'Stopped', 'isDataAvailable': False, 'isTriggerActive': False}

✅ Query completed successfully!
   Final status: {'message': 'Stopped', 'isDataAvailable': False, 'isTriggerActive': False}


### Verify the Results

Read the bronze zone table to see the processed data:

In [61]:
df = spark.sql('select * from silver.shipment')
df.show()

+--------------------+----------------+-----------+--------------+-----------------------+-------------------+--------------------+--------------------+------------------+--------------------+----------------+-------------------+-----------------------+--------------------+---------------------+-----------------+--------------------+------------------+--------------------+------+----------+------------+-----------------------------+------+--------------------+---------------+---------------+--------------------+-------------+--------------+------------------+---------------+----------------+------------+---------------+----------------------+-------------+--------------------+--------------------+--------------------+--------------------+------+------------------+-----+---------------------+
|       enqueued_time|        producer|record_type|schema_version|committed_delivery_date|current_facility_id|         customer_id|       customer_name|    declared_value| destination_address|desti

In [91]:
spark.sql('drop schema bronze cascade')
spark.sql('drop schema silver cascade')


DataFrame[]

In [23]:
df = spark.sql('select * from silver.order')
df.show()

+--------------------+--------------------+-------------------+-----------+------+--------------------+---------------+--------------------+--------------------+--------------+--------------------+-----------+------------------+--------+--------------------+----------+-----------------+---------------------+
|            order_id|        order_number|         order_date|order_total|source|         customer_id|organization_id|        generated_at|         description|extended_price|             item_id|line_number|        net_weight|quantity|                 sku|unit_price|warranty_included|_processing_timestamp|
+--------------------+--------------------+-------------------+-----------+------+--------------------+---------------+--------------------+--------------------+--------------+--------------------+-----------+------------------+--------+--------------------+----------+-----------------+---------------------+
|eb7c44a2-f6d1-42e...|SO127482402507682...|2025-11-01 00:00:00|    880